# Build a Transformer: Name Generation

In this notebook we build a **character-level transformer** for generating names, step by step.
We start from the simplest possible model (a bigram count table) and work our way up to a full
transformer with multi-head self-attention, residual connections, and layer normalization.

This notebook accompanies the lesson on the [mlearn website](https://mlearn.dev).

**Models we will build:**
1. Bigram (counting)
2. Neural Bigram (single linear layer)
3. MLP Language Model
4. Single-Head / Multi-Head Self-Attention
5. Full Transformer

---

**How this notebook works:**

Each section has:
1. A **markdown explanation** of what to build
2. An **exercise cell** with skeleton code and `# YOUR CODE HERE` placeholders
3. A **collapsed solution cell** you can expand if you get stuck

Try to write the code yourself first, then check the solution!

## 1. Setup

Run this cell to install dependencies and import libraries. Nothing to change here.

> **Python for JS devs:** Python uses `import` like ES6 `import`, but packages are installed globally via `pip` (like `npm install -g`). The `as` keyword renames imports: `import matplotlib.pyplot as plt` is like `const plt = require('matplotlib').pyplot`. Indentation defines blocks (no curly braces).

In [ ]:
!pip install torch matplotlib -q

import torch                         # main tensor/math library (no JS equivalent -- closest is tf.js)
import torch.nn as nn                # neural network layers module
import torch.nn.functional as F      # stateless functions (softmax, relu, cross_entropy, etc.)
import matplotlib.pyplot as plt      # plotting (like Chart.js or D3)
import random                        # Python built-in random (like Math.random() with more utilities)

# reproducibility
torch.manual_seed(42)
random.seed(42)

print(f'PyTorch version: {torch.__version__}')  # f-string: like JS template literal `PyTorch version: ${torch.__version__}`

## 2. Dataset & Character Encoding

We use a hardcoded list of common names. Each name is a sequence of characters;
we add a special `'.'` token to mark the start and end of a name.

Run the next two cells as-is -- they set up the data we will use throughout.

In [ ]:
names = [
    "emma", "olivia", "ava", "sophia", "isabella", "mia", "charlotte", "amelia",
    "harper", "evelyn", "abigail", "emily", "elizabeth", "sofia", "ella", "madison",
    "scarlett", "victoria", "aria", "grace", "chloe", "camila", "penelope", "riley",
    "layla", "lillian", "nora", "zoey", "mila", "aubrey", "hannah", "lily", "addison",
    "eleanor", "natalie", "luna", "savannah", "brooklyn", "leah", "zoe", "stella",
    "hazel", "ellie", "paisley", "audrey", "skylar", "violet", "claire", "bella",
    "aurora", "liam", "noah", "oliver", "james", "elijah", "william", "henry",
    "lucas", "benjamin", "theodore", "jack", "levi", "alexander", "mason", "ethan",
    "daniel", "jacob", "michael", "sebastian", "owen", "aiden", "samuel", "ryan",
    "nathan", "caleb", "christian", "dylan", "landon", "josiah", "andrew", "david",
    "adrian", "miles", "eli", "nolan", "hunter", "isaiah", "thomas", "aaron",
    "roman", "colton", "leo", "connor", "ezekiel", "hudson", "easton", "asher",
    "robert", "xavier", "adam", "jose", "ivan", "clara", "elena", "iris"
]

print(f'Number of names: {len(names)}')  # len() is like .length
print(f'Examples: {names[:10]}')  # [:10] slicing -- like .slice(0, 10)

> **Python for JS devs:** The next cell uses *dictionary comprehensions* (`{k: v for ...}`) and `enumerate()`. Think of `{ch: i for i, ch in enumerate(chars)}` as `Object.fromEntries(chars.map((ch, i) => [ch, i]))`. Also, `.items()` on a dict returns key-value pairs like `Object.entries()`.

In [ ]:
# Build character vocabulary
chars = sorted(list(set(''.join(names))))  # set() = unique values (like new Set()), sorted() = Array.from(...).sort()
chars = ['.'] + chars  # list concatenation with + (like [...arr1, ...arr2])
vocab_size = len(chars)

# Dict comprehensions: like Object.fromEntries(chars.map((ch, i) => [ch, i]))
stoi = {ch: i for i, ch in enumerate(chars)}  # enumerate() yields [index, value] pairs
itos = {i: ch for ch, i in stoi.items()}  # .items() is like Object.entries() -- returns [key, value] pairs

print(f'Vocabulary ({vocab_size} chars): {chars}')
print(f'stoi example: a -> {stoi["a"]}, z -> {stoi["z"]}')

In [ ]:
# Train / validation split
random.shuffle(names)  # in-place shuffle (mutates the list, like arr.sort(() => Math.random() - 0.5) but proper)
n = int(0.9 * len(names))  # int() truncates float to integer (like Math.floor())
train_names = names[:n]   # slice from start to n (like .slice(0, n))
val_names = names[n:]     # slice from n to end (like .slice(n))
print(f'Train: {len(train_names)}, Val: {len(val_names)}')

## 3. Bigram Model (Counting)

The simplest language model: just count how often character *b* follows character *a*,
then normalize to get probabilities.

### Exercise 1: Build the Bigram Count Matrix

**Goal:** Create a matrix `N` of shape `(vocab_size, vocab_size)` where `N[i, j]` counts how many times character `j` follows character `i` in the training data.

**Steps:**
1. Initialize `N` as a zero matrix of integers with shape `(vocab_size, vocab_size)`
2. For each name in `train_names`, wrap it with `'.'` tokens: `['.'] + list(name) + ['.']`
3. Loop over consecutive character pairs and increment `N[stoi[ch1], stoi[ch2]]`

**Expected output:** Count matrix shape: `torch.Size([27, 27])`, Total bigrams: ~500

In [ ]:
# Exercise 1: Build bigram count matrix
N = torch.zeros((vocab_size, vocab_size), dtype=torch.int32)  # 2D zero matrix (like a 2D array filled with 0s)

for name in train_names:  # for...of in JS
    # Wrap name with start/end tokens: ['.'] + list(name) + ['.']
    # list(name) splits string into chars -- like name.split('')
    # YOUR CODE HERE
    chs = None  # replace with the wrapped character list

    # Loop over consecutive pairs and increment counts
    # zip(a, b) pairs elements from two lists -- like lodash _.zip(a, b)
    # YOUR CODE HERE
    pass  # pass = placeholder for empty block (JS doesn't need this)

print('Count matrix shape:', N.shape)  # .shape is like [rows, cols] dimensions
print('Total bigrams:', N.sum().item())  # .item() extracts a plain Python number from a 1-element tensor

In [ ]:
#@title Solution (click to expand)
# Build bigram count matrix
N = torch.zeros((vocab_size, vocab_size), dtype=torch.int32)

for name in train_names:
    chs = ['.'] + list(name) + ['.']  # list('abc') = ['a','b','c'] -- like 'abc'.split('')
    for ch1, ch2 in zip(chs, chs[1:]):  # zip pairs elements; chs[1:] is like chs.slice(1)
        # Tuple unpacking: ch1, ch2 = pair -- like const [ch1, ch2] = pair
        N[stoi[ch1], stoi[ch2]] += 1  # += works on tensor elements just like JS

print('Count matrix shape:', N.shape)
print('Total bigrams:', N.sum().item())

### Visualize the Bigram Counts

Run this cell to see a heatmap of the bigram counts. No exercise here -- just observe the patterns.

In [ ]:
# Visualize the bigram counts
fig, ax = plt.subplots(figsize=(16, 16))  # tuple unpacking: like const [fig, ax] = ...
ax.imshow(N.float(), cmap='Blues')  # .float() converts int tensor to float tensor
for i in range(vocab_size):  # range(n) is like Array(n).keys() or for(let i=0; i<n; i++)
    for j in range(vocab_size):
        label = itos[i] + itos[j]  # string concatenation (same as JS)
        ax.text(j, i, label, ha='center', va='bottom', fontsize=6, color='gray')
        ax.text(j, i, N[i, j].item(), ha='center', va='top', fontsize=6)
ax.set_title('Bigram Counts')
plt.tight_layout()
plt.show()

### Exercise 2: Compute Bigram Probabilities and Validation Loss

**Goal:** Convert the count matrix into a probability matrix and evaluate it on the validation set.

**Steps:**
1. Add 1 to all counts (Laplace smoothing to avoid zero probabilities), convert to float
2. Normalize each row so it sums to 1 (divide by row sums, keeping dimensions)
3. Compute the negative log-likelihood (NLL) on `val_names`:
   - For each bigram `(ch1, ch2)`, look up `P[stoi[ch1], stoi[ch2]]` and take its log
   - Average the negative log-likelihood over all bigrams

**Expected output:** `bigram_count_loss` should be around 2.5-3.0

In [ ]:
# Exercise 2: Probability matrix and validation loss

# Step 1: Add smoothing and convert to float
P = # YOUR CODE HERE

# Step 2: Normalize each row to sum to 1
# dim=1 means "along columns" (each row sums to 1); keepdim=True preserves shape for broadcasting
P = # YOUR CODE HERE

# Step 3: Compute NLL on validation set
log_likelihood = 0.0
count = 0
for name in val_names:
    chs = ['.'] + list(name) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        # Look up the probability and add its log
        # torch.log() is like Math.log()
        # YOUR CODE HERE
        pass

bigram_count_loss = -log_likelihood / count
print(f'Bigram (count) val NLL: {bigram_count_loss.item():.4f}')  # :.4f = format to 4 decimal places

In [ ]:
#@title Solution (click to expand)
# Probability matrix (add smoothing to avoid zero probs)
P = (N + 1).float()  # broadcast: adds 1 to every element, then converts to float
P = P / P.sum(dim=1, keepdim=True)  # normalize rows; keepdim=True keeps the shape for broadcasting

# Compute negative log-likelihood on val set
log_likelihood = 0.0  # 0.0 is a float literal (Python distinguishes int vs float)
count = 0
for name in val_names:
    chs = ['.'] + list(name) + ['.']
    for ch1, ch2 in zip(chs, chs[1:]):
        prob = P[stoi[ch1], stoi[ch2]]  # 2D indexing: like P[row][col] but in one expression
        log_likelihood += torch.log(prob)  # += works with tensors too
        count += 1

bigram_count_loss = -log_likelihood / count
print(f'Bigram (count) val NLL: {bigram_count_loss.item():.4f}')

### Exercise 3: Generate Names from the Bigram Model

**Goal:** Sample names character-by-character using the probability matrix `P`.

**Steps:**
1. Start with index 0 (the `'.'` token)
2. Use `torch.multinomial(P[ix], ...)` to sample the next character
3. Stop when you sample index 0 again (end token)
4. Convert indices back to characters using `itos`

In [ ]:
# Exercise 3: Generate names from the bigram count model
g = torch.Generator().manual_seed(42)  # seeded RNG (like a seedable Math.random)

print('Bigram (count) generated names:')
for _ in range(10):  # _ is a throwaway variable -- we don't need the loop index (like for(let _=0; _<10; _++))
    out = []  # empty list (like [])
    ix = 0  # start token
    while True:  # like while(true) in JS
        # Sample next character index from P[ix]
        # torch.multinomial draws random samples weighted by probabilities
        # YOUR CODE HERE
        # ix = ...

        if ix == 0:  # Python uses == for comparison (same as JS), but no === needed
            break
        out.append(itos[ix])  # .append() is like .push()
    print('  ' + ''.join(out))  # ''.join(list) is like list.join('') in JS (note: separator comes first)

In [ ]:
#@title Solution (click to expand)
# Generate names from the bigram count model
g = torch.Generator().manual_seed(42)

print('Bigram (count) generated names:')
for _ in range(10):  # _ = throwaway variable, we just want 10 iterations
    out = []
    ix = 0  # start token
    while True:
        p = P[ix]  # get probability row for current character
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()  # .item() unwraps scalar tensor to plain int
        if ix == 0:
            break
        out.append(itos[ix])  # .append() is like .push()
    print('  ' + ''.join(out))  # ''.join() is like Array.join('') but called on the separator

## 4. Neural Bigram Model

Replace the explicit count table with a single linear layer.
The input is a one-hot vector (or equivalently, an embedding lookup),
and the output is logits over the next character.

First, we build the dataset. Run this cell as-is.

> **Python for JS devs:** `def` declares a function (like `function` keyword). Python returns tuples with comma-separated values (`return a, b`) which you unpack on the caller side (`x, y = func()`) -- like `const [x, y] = func()` in JS. `torch.tensor(list)` converts a plain Python list to a typed tensor.

In [ ]:
# Build training data for bigram: (prev_char) -> (next_char)
def build_bigram_dataset(name_list):  # def = function declaration; type hints are optional in Python
    xs, ys = [], []  # tuple unpacking: like const [xs, ys] = [[], []]
    for name in name_list:
        chs = ['.'] + list(name) + ['.']
        for ch1, ch2 in zip(chs, chs[1:]):
            xs.append(stoi[ch1])
            ys.append(stoi[ch2])
    return torch.tensor(xs), torch.tensor(ys)  # returns a tuple (like returning [a, b] in JS)

xtr_bi, ytr_bi = build_bigram_dataset(train_names)  # unpack returned tuple: like const [xtr, ytr] = ...
xval_bi, yval_bi = build_bigram_dataset(val_names)
print(f'Train bigrams: {len(xtr_bi)}, Val bigrams: {len(xval_bi)}')

### Exercise 4: Train the Neural Bigram Model

**Goal:** Train a single weight matrix `W` of shape `(vocab_size, vocab_size)` so that `one_hot(x) @ W` produces logits for the next character.

**Forward pass steps:**
1. One-hot encode `xtr_bi` to shape `(N, vocab_size)` using `F.one_hot(...).float()`
2. Compute `logits = xenc @ W` -- shape `(N, vocab_size)`
3. Compute probabilities: `exp(logits)` then normalize rows
4. Compute loss: `-log(probs[i, ytr_bi[i]])` averaged over all examples
5. Add L2 regularization: `0.01 * (W ** 2).mean()`

**Backward pass:** Zero grad, call `.backward()`, update `W.data -= 50 * W.grad`

**Expected output:** Loss should decrease from ~3.7 to ~2.6 over 200 steps

In [ ]:
# Exercise 4: Neural bigram model
W = torch.randn((vocab_size, vocab_size), requires_grad=True, generator=torch.Generator().manual_seed(42))
# requires_grad=True tells PyTorch to track operations for automatic differentiation (autograd)

# Training loop
for step in range(200):  # range(200) = 0,1,2,...,199
    # Forward pass
    # Step 1: One-hot encode the inputs
    xenc = F.one_hot(xtr_bi, num_classes=vocab_size).float()  # one-hot: [2] -> [0,0,1,0,...,0]

    # Step 2: Compute logits
    # @ is matrix multiplication (like numpy.matmul, no JS equivalent)
    # YOUR CODE HERE
    logits = None

    # Step 3: Compute probabilities (exp + normalize)
    # YOUR CODE HERE
    counts = None
    probs = None

    # Step 4: Compute negative log-likelihood loss
    # torch.arange(n) creates [0,1,2,...,n-1] -- like Array.from({length:n}, (_,i) => i)
    # Advanced indexing: probs[rows, cols] picks one element per row
    # YOUR CODE HERE
    loss = None

    # Step 5: Add regularization
    # ** is exponentiation (like Math.pow() or ** in JS)
    # YOUR CODE HERE

    # Backward pass
    W.grad = None  # reset gradients (None is like null in JS; Python has no undefined)
    loss.backward()  # autograd: computes d(loss)/d(W) and stores it in W.grad

    # Update weights
    W.data -= 50 * W.grad  # .data bypasses autograd; -= is in-place subtraction (same as JS)

    if step % 50 == 0:  # % is modulo (same as JS)
        print(f'Step {step:4d} | loss: {loss.item():.4f}')  # :4d = pad to 4 digits

# Validation loss
with torch.no_grad():  # context manager: disables gradient tracking inside this block (saves memory)
    xenc = F.one_hot(xval_bi, num_classes=vocab_size).float()
    logits = xenc @ W
    neural_bigram_loss = F.cross_entropy(logits, yval_bi)
    print(f'\nNeural bigram val loss: {neural_bigram_loss.item():.4f}')

In [ ]:
#@title Solution (click to expand)
# Neural bigram: a single linear layer (logits = W[input_char])
W = torch.randn((vocab_size, vocab_size), requires_grad=True, generator=torch.Generator().manual_seed(42))
# requires_grad=True: opt into autograd so PyTorch tracks gradients for this tensor

# Training loop
for step in range(200):
    # forward
    xenc = F.one_hot(xtr_bi, num_classes=vocab_size).float()
    logits = xenc @ W  # @ = matrix multiply operator
    counts = logits.exp()  # element-wise e^x (like Math.exp() but for whole tensor)
    probs = counts / counts.sum(dim=1, keepdim=True)  # normalize each row to sum to 1
    loss = -probs[torch.arange(len(ytr_bi)), ytr_bi].log().mean()
    # ^ advanced indexing: picks probs[0,ytr[0]], probs[1,ytr[1]], etc. -- then log + mean
    # add regularization
    loss += 0.01 * (W ** 2).mean()  # ** is power operator (same as JS **)

    # backward
    W.grad = None  # None is like null -- clears old gradients
    loss.backward()  # autograd: fills W.grad with d(loss)/d(W)

    # update
    W.data -= 50 * W.grad  # manual SGD step; .data bypasses autograd tracking

    if step % 50 == 0:
        print(f'Step {step:4d} | loss: {loss.item():.4f}')

# Validation loss
with torch.no_grad():  # disables gradient computation (like a performance optimization scope)
    xenc = F.one_hot(xval_bi, num_classes=vocab_size).float()
    logits = xenc @ W
    neural_bigram_loss = F.cross_entropy(logits, yval_bi)  # combines softmax + NLL in one step
    print(f'\nNeural bigram val loss: {neural_bigram_loss.item():.4f}')

### Exercise 5: Generate Names from the Neural Bigram

**Goal:** Sample names using the learned weight matrix `W`.

**Steps (inside the while loop):**
1. One-hot encode the current index `ix` as a `(1, vocab_size)` tensor
2. Compute logits: `xenc @ W`
3. Convert to probabilities with `F.softmax(logits, dim=1)`
4. Sample from the distribution with `torch.multinomial`

In [ ]:
# Exercise 5: Generate names from the neural bigram model
g = torch.Generator().manual_seed(42)
print('Neural bigram generated names:')
for _ in range(10):
    out = []
    ix = 0
    while True:
        # One-hot encode current character, compute logits, get probabilities, sample
        # YOUR CODE HERE
        # xenc = ...
        # logits = ...
        # p = ...
        # ix = ...

        if ix == 0:
            break
        out.append(itos[ix])
    print('  ' + ''.join(out))

In [ ]:
#@title Solution (click to expand)
# Generate names from the neural bigram model
g = torch.Generator().manual_seed(42)
print('Neural bigram generated names:')
for _ in range(10):
    out = []
    ix = 0
    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes=vocab_size).float()  # wrap ix in list to make 1D tensor
        logits = xenc @ W
        p = F.softmax(logits, dim=1)  # softmax converts logits to probabilities (like softmax in tf.js)
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        if ix == 0:
            break
        out.append(itos[ix])
    print('  ' + ''.join(out))

## 5. MLP Language Model

Now we look at more context than just one character. We use a context window
of `block_size` characters, embed them, concatenate, and pass through a
hidden layer with tanh activation.

First, let's build the dataset. Run this cell as-is.

In [ ]:
block_size = 3  # context length: how many previous chars to look at

def build_mlp_dataset(name_list):
    X, Y = [], []  # tuple unpacking assignment
    for name in name_list:
        context = [0] * block_size  # [0, 0, 0] -- like new Array(3).fill(0)
        for ch in name + '.':  # iterate over each character; + concatenates strings
            ix = stoi[ch]
            X.append(context[:])  # context[:] makes a shallow copy (like [...context])
            Y.append(ix)
            context = context[1:] + [ix]  # slide window: drop first, append new (like [...context.slice(1), ix])
    return torch.tensor(X), torch.tensor(Y)

Xtr, Ytr = build_mlp_dataset(train_names)
Xval, Yval = build_mlp_dataset(val_names)
print(f'Train: X={Xtr.shape}, Y={Ytr.shape}')  # .shape is a tuple of dimensions (like [rows, cols])
print(f'Val:   X={Xval.shape}, Y={Yval.shape}')

# Show a few examples
for i in range(5):
    ctx = ''.join(itos[c.item()] for c in Xtr[i])  # generator expression: like Xtr[i].map(c => itos[c]).join('')
    print(f'  {ctx} --> {itos[Ytr[i].item()]}')

### Exercise 6: Initialize MLP Parameters

**Goal:** Create the parameters for a 2-layer MLP: embedding `C`, hidden layer `(W1, b1)`, output layer `(W2, b2)`.

**Architecture:**
- `C`: Embedding table of shape `(vocab_size, n_embd)` -- maps each character to a 10-d vector
- `W1`: shape `(block_size * n_embd, n_hidden)` -- takes concatenated embeddings to hidden layer
- `b1`: shape `(n_hidden,)` -- hidden layer bias
- `W2`: shape `(n_hidden, vocab_size)` -- hidden to output logits
- `b2`: shape `(vocab_size,)` -- output bias

**Initialization tips:**
- `W1` should use Kaiming-like init: multiply by `(5/3) / sqrt(block_size * n_embd)`
- `W2` should be small: multiply by `0.01`
- `b1` should be small: multiply by `0.01`
- `b2` should be zero: multiply by `0`
- All parameters need `requires_grad = True`

In [ ]:
# Exercise 6: MLP model parameters
n_embd = 10
n_hidden = 200

g = torch.Generator().manual_seed(42)

# YOUR CODE HERE: Initialize C, W1, b1, W2, b2
C = None   # (vocab_size, n_embd)
W1 = None  # (block_size * n_embd, n_hidden), scaled by (5/3)/sqrt(fan_in)
b1 = None  # (n_hidden,), small
W2 = None  # (n_hidden, vocab_size), small
b2 = None  # (vocab_size,), zero

parameters_mlp = [C, W1, b1, W2, b2]
for p in parameters_mlp:
    p.requires_grad = True

print(f'MLP parameters: {sum(p.nelement() for p in parameters_mlp)}')

In [ ]:
#@title Solution (click to expand)
# MLP model
n_embd = 10
n_hidden = 200

g = torch.Generator().manual_seed(42)
C = torch.randn((vocab_size, n_embd), generator=g)  # embedding table: each row is a char's vector
W1 = torch.randn((block_size * n_embd, n_hidden), generator=g) * (5/3) / (block_size * n_embd)**0.5  # Kaiming init
b1 = torch.randn(n_hidden, generator=g) * 0.01
W2 = torch.randn((n_hidden, vocab_size), generator=g) * 0.01
b2 = torch.randn(vocab_size, generator=g) * 0  # multiply by 0 = zeros, but keeps randn call for consistency

parameters_mlp = [C, W1, b1, W2, b2]  # plain Python list of tensors
for p in parameters_mlp:
    p.requires_grad = True  # enable gradient tracking on each parameter

print(f'MLP parameters: {sum(p.nelement() for p in parameters_mlp)}')  # generator expr inside sum()

### Exercise 7: MLP Training Loop

**Goal:** Train the MLP for 10000 steps with mini-batches of size 64.

**Forward pass for each mini-batch:**
1. Look up embeddings: `emb = C[Xb]` gives shape `(B, block_size, n_embd)`
2. Flatten: reshape `emb` to `(B, block_size * n_embd)`
3. Hidden layer: `h = tanh(flat @ W1 + b1)`
4. Output: `logits = h @ W2 + b2`
5. Loss: `F.cross_entropy(logits, Yb)`

**Learning rate schedule:** `0.1` for steps 0-4999, `0.01` for steps 5000-9999

**Expected output:** Loss should decrease from ~3.5 to ~2.2

In [ ]:
# Exercise 7: Training loop
mlp_losses = []
for step in range(10000):
    # Mini-batch
    ix = torch.randint(0, Xtr.shape[0], (64,))  # random indices; (64,) is a 1-element tuple for shape
    Xb, Yb = Xtr[ix], Ytr[ix]  # index with tensor to select rows (like fancy indexing)

    # Forward pass
    # Step 1: Embedding lookup
    emb = C[Xb]  # (B, block_size, n_embd) -- indexing a tensor with indices = embedding lookup

    # Step 2-4: Flatten, hidden layer with tanh, output logits
    # .view() reshapes a tensor (like changing array dimensions) -- similar to numpy reshape
    # -1 means "infer this dimension" (like saying "whatever fits")
    # YOUR CODE HERE
    h = None     # hidden activations
    logits = None  # output logits

    # Step 5: Compute loss
    loss = F.cross_entropy(logits, Yb)  # cross_entropy = softmax + negative log-likelihood

    # Backward pass
    for p in parameters_mlp:
        p.grad = None  # clear gradients (None is like null/undefined)
    loss.backward()  # autograd: computes gradients for all requires_grad=True tensors

    # Update with learning rate schedule
    lr = 0.1 if step < 5000 else 0.01  # ternary expression: like step < 5000 ? 0.1 : 0.01
    for p in parameters_mlp:
        p.data -= lr * p.grad  # SGD update; .data avoids tracking this op in autograd

    if step % 2000 == 0:
        print(f'Step {step:5d} | loss: {loss.item():.4f}')
    mlp_losses.append(loss.item())

# Validation loss
with torch.no_grad():  # disables gradient tracking in this block
    emb = C[Xval]
    h = torch.tanh(emb.view(-1, block_size * n_embd) @ W1 + b1)  # .view() reshapes, @ = matmul
    logits = h @ W2 + b2
    mlp_val_loss = F.cross_entropy(logits, Yval)
    print(f'\nMLP val loss: {mlp_val_loss.item():.4f}')

In [ ]:
#@title Solution (click to expand)
# Training loop
mlp_losses = []
for step in range(10000):
    # mini-batch
    ix = torch.randint(0, Xtr.shape[0], (64,))  # random batch of 64 indices
    Xb, Yb = Xtr[ix], Ytr[ix]  # fancy indexing: select rows by index tensor

    # forward
    emb = C[Xb]  # (B, block_size, n_embd) -- embedding lookup via indexing
    h = torch.tanh(emb.view(-1, block_size * n_embd) @ W1 + b1)  # .view(-1, n) flattens to 2D; @ = matmul
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, Yb)

    # backward
    for p in parameters_mlp:
        p.grad = None  # reset gradients (None = null)
    loss.backward()  # autograd computes all gradients

    # update
    lr = 0.1 if step < 5000 else 0.01  # Python ternary: val_if_true if condition else val_if_false
    for p in parameters_mlp:
        p.data -= lr * p.grad  # in-place SGD; .data skips autograd

    if step % 2000 == 0:
        print(f'Step {step:5d} | loss: {loss.item():.4f}')
    mlp_losses.append(loss.item())

# Validation loss
with torch.no_grad():
    emb = C[Xval]
    h = torch.tanh(emb.view(-1, block_size * n_embd) @ W1 + b1)
    logits = h @ W2 + b2
    mlp_val_loss = F.cross_entropy(logits, Yval)
    print(f'\nMLP val loss: {mlp_val_loss.item():.4f}')

### Exercise 8: Generate Names from the MLP

**Goal:** Sample names by repeatedly feeding the last `block_size` characters through the MLP.

**Steps (inside the while loop):**
1. Look up embeddings for the current context: `C[torch.tensor([context])]`
2. Flatten and pass through hidden layer: `tanh(flat @ W1 + b1)`
3. Compute logits: `h @ W2 + b2`
4. Convert to probabilities with softmax
5. Sample and update the context window (slide by 1)

In [ ]:
# Exercise 8: Generate names from the MLP
g = torch.Generator().manual_seed(42)
print('MLP generated names:')
for _ in range(10):  # _ = don't care about the loop variable
    out = []
    context = [0] * block_size  # [0, 0, 0] initial context
    while True:
        # YOUR CODE HERE: embed, flatten, hidden layer, logits, softmax, sample
        # emb = C[torch.tensor([context])]  -- wrap in list to add batch dimension
        # .view(1, -1) flattens to (1, block_size*n_embd) -- like .reshape() / .flat()
        # emb = ...
        # h = ...
        # logits = ...
        # p = ...
        # ix = ...

        if ix == 0:
            break
        out.append(itos[ix])
        context = context[1:] + [ix]  # slide window: drop oldest, add newest (like [...ctx.slice(1), ix])
    print('  ' + ''.join(out))

In [ ]:
#@title Solution (click to expand)
# Generate names from the MLP
g = torch.Generator().manual_seed(42)
print('MLP generated names:')
for _ in range(10):
    out = []
    context = [0] * block_size
    while True:
        emb = C[torch.tensor([context])]  # [context] adds batch dim: shape (1, block_size, n_embd)
        h = torch.tanh(emb.view(1, -1) @ W1 + b1)  # .view(1, -1) flattens last 2 dims
        logits = h @ W2 + b2
        p = F.softmax(logits, dim=1)
        ix = torch.multinomial(p, num_samples=1, generator=g).item()  # sample 1 index from distribution
        if ix == 0:
            break
        out.append(itos[ix])
        context = context[1:] + [ix]  # slide window right
    print('  ' + ''.join(out))

## 6. Self-Attention

Before building the full transformer, let us understand self-attention.
Each position produces a **query** and a **key**; the dot product of queries and keys
determines how much each position attends to every other. A **causal mask** ensures
we only look at the past.

### Exercise 9: Single-Head Self-Attention

**Goal:** Implement the core self-attention mechanism from scratch.

**Given:** Random input `x` of shape `(B, T, C_dim)` = `(4, 3, 10)`, and `head_size = 16`.

**Steps:**
1. Create three `nn.Linear` layers (no bias): `key`, `query`, `value` -- each maps `C_dim -> head_size`
2. Compute `k = key(x)`, `q = query(x)`, `v = value(x)` -- all shape `(B, T, head_size)`
3. Compute attention scores: `wei = q @ k.transpose(-2, -1) * head_size**-0.5` -- shape `(B, T, T)`
4. Apply causal mask: create a lower-triangular matrix, mask upper triangle to `-inf`
5. Apply softmax on the last dimension
6. Compute output: `out = wei @ v` -- shape `(B, T, head_size)`

**Why scale by `head_size**-0.5`?** Without scaling, the dot products grow large with `head_size`, pushing softmax into saturation. Scaling keeps the variance roughly 1.

> **Python for JS devs:** `nn.Module` is PyTorch's base class for neural network layers -- think of it like `class MyLayer extends Component`. You define layers in `__init__` (the constructor) and the computation in `forward()` (called when you do `layer(input)`). `self` is Python's explicit `this` -- you must include it as the first method parameter. `super().__init__()` is exactly like `super()` in a JS constructor.

In [ ]:
# Exercise 9: Single-head self-attention
torch.manual_seed(42)

B, T, C_dim = 4, block_size, n_embd  # batch=4, time=3, channels=10
# Multiple assignment: like const [B, T, C_dim] = [4, block_size, n_embd]
head_size = 16

# Random input (pretend these are embeddings)
x = torch.randn(B, T, C_dim)

# Step 1: Create key, query, value linear projections (no bias)
# nn.Linear(in, out, bias=False) is a learned weight matrix -- like a dense/fully-connected layer
# YOUR CODE HERE
key = None    # nn.Linear(C_dim, head_size, bias=False)
query = None  # nn.Linear(C_dim, head_size, bias=False)
value = None  # nn.Linear(C_dim, head_size, bias=False)

# Step 2: Project input to k, q, v
# YOUR CODE HERE
k = None  # (B, T, head_size)
q = None  # (B, T, head_size)
v = None  # (B, T, head_size)

# Step 3: Compute scaled attention scores
# .transpose(-2, -1) swaps the last two dims -- like transposing a matrix
# Hint: q @ k.transpose(-2, -1) gives (B, T, T), then scale by head_size**-0.5
# YOUR CODE HERE
wei = None  # (B, T, T)

# Step 4: Apply causal mask
# torch.tril = lower-triangular matrix of ones (mask so position i can only see positions <= i)
# YOUR CODE HERE
tril = torch.tril(torch.ones(T, T))
wei = None  # masked version -- use .masked_fill(condition, value)

# Step 5: Softmax
# dim=-1 means apply along the last dimension (each row sums to 1)
# YOUR CODE HERE
wei = None

# Step 6: Weighted aggregation of values
# YOUR CODE HERE
out = None  # (B, T, head_size)

print('Attention weights (first example):')
print(wei[0].detach())  # .detach() removes from autograd graph (for safe printing)
print(f'\nOutput shape: {out.shape}')

In [ ]:
#@title Solution (click to expand)
# Demonstrate single-head self-attention on a small example
torch.manual_seed(42)

B, T, C_dim = 4, block_size, n_embd  # batch, time, channels
head_size = 16

# Random input (pretend these are embeddings)
x = torch.randn(B, T, C_dim)

# nn.Linear creates a learnable weight matrix + optional bias
key = nn.Linear(C_dim, head_size, bias=False)
query = nn.Linear(C_dim, head_size, bias=False)
value = nn.Linear(C_dim, head_size, bias=False)

k = key(x)    # (B, T, head_size) -- calling a module runs its forward() method
q = query(x)  # (B, T, head_size)
v = value(x)  # (B, T, head_size)

# Attention scores
wei = q @ k.transpose(-2, -1) * head_size**-0.5  # (B, T, T) -- scale prevents softmax saturation
# .transpose(-2, -1) swaps last two dims (like transposing a 2D matrix within each batch)

# Causal mask: can only attend to current and previous positions
tril = torch.tril(torch.ones(T, T))  # lower-triangular matrix of 1s
wei = wei.masked_fill(tril == 0, float('-inf'))  # fill upper triangle with -inf (softmax turns these to 0)
wei = F.softmax(wei, dim=-1)  # dim=-1 = softmax along last axis

out = wei @ v  # (B, T, head_size) -- weighted sum of values

print('Attention weights (first example):')
print(wei[0].detach())
print(f'\nOutput shape: {out.shape}')

### Exercise 10: Multi-Head Attention Module

**Goal:** Implement `Head` and `MultiHeadAttention` as `nn.Module` classes.

**`Head` class:**
- `__init__`: Create `key`, `query`, `value` linear layers (no bias), each `n_embd -> head_size`. Register a buffer `tril` = lower-triangular matrix of shape `(block_size, block_size)`.
- `forward(x)`: Given `x` of shape `(B, T, C)`, compute k, q, v, attention weights (scaled + masked + softmax), and return `wei @ v`.

**`MultiHeadAttention` class:**
- `__init__`: Create a `ModuleList` of `n_heads` `Head` instances, plus a projection layer `nn.Linear(n_heads * head_size, n_embd)`.
- `forward(x)`: Run each head on `x`, concatenate outputs along the last dim, project back to `n_embd`.

**Expected output:** Input `(2, 3, 10)` -> Output `(2, 3, 10)` (same shape, since projection maps back)

In [ ]:
# Exercise 10: Multi-head attention

class Head(nn.Module):  # class inherits from nn.Module (like "class Head extends nn.Module" in JS/TS)
    """One head of self-attention."""
    def __init__(self, n_embd, head_size, block_size):  # __init__ = constructor (like constructor() in JS)
        super().__init__()  # must call parent constructor (like super() in JS)
        # YOUR CODE HERE: create key, query, value linear layers (no bias)
        self.key = None    # self.x is like this.x in JS -- instance attributes
        self.query = None
        self.value = None
        # Register the causal mask as a buffer (not a parameter -- won't be updated by optimizer)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):  # forward() is called when you do module(input) -- like a __call__ method
        B, T, C = x.shape  # unpack shape tuple: like const [B, T, C] = x.shape
        # YOUR CODE HERE: compute k, q, v
        k = None
        q = None
        v = None
        # YOUR CODE HERE: compute attention weights (scale, mask, softmax)
        wei = None  # q @ k^T scaled by head_size**-0.5
        wei = None  # apply causal mask (use self.tril[:T, :T] -- slice to current sequence length)
        wei = None  # softmax
        # YOUR CODE HERE: return weighted sum
        return None  # wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_heads, head_size, block_size):
        super().__init__()
        # YOUR CODE HERE: create ModuleList of Heads and projection layer
        # nn.ModuleList is like an array of modules that PyTorch knows about (registers their params)
        self.heads = None  # nn.ModuleList([Head(...) for _ in range(n_heads)])
        self.proj = None   # nn.Linear(n_heads * head_size, n_embd)

    def forward(self, x):
        # YOUR CODE HERE: run all heads, concatenate, project
        # torch.cat([...], dim=-1) concatenates along last dimension
        out = None  # concatenated head outputs: [h(x) for h in self.heads] is a list comprehension
        return None  # projected output

# Quick test
mha = MultiHeadAttention(n_embd=10, n_heads=2, head_size=8, block_size=3)
test_in = torch.randn(2, 3, 10)
test_out = mha(test_in)
print(f'Multi-head attention: input {test_in.shape} -> output {test_out.shape}')

In [ ]:
#@title Solution (click to expand)
# Multi-head attention: run several heads in parallel and concatenate

class Head(nn.Module):
    """One head of self-attention."""
    def __init__(self, n_embd, head_size, block_size):
        super().__init__()  # always call super().__init__() first in nn.Module subclasses
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):  # called automatically when you do head(x)
        B, T, C = x.shape
        k = self.key(x)    # self.key(x) calls forward() of the Linear layer
        q = self.query(x)
        v = self.value(x)
        wei = q @ k.transpose(-2, -1) * k.shape[-1]**-0.5  # .shape[-1] = last dim size
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))  # [:T, :T] slices the mask
        wei = F.softmax(wei, dim=-1)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, n_embd, n_heads, head_size, block_size):
        super().__init__()
        # List comprehension inside ModuleList: like Array(n_heads).fill(null).map(() => new Head(...))
        self.heads = nn.ModuleList([Head(n_embd, head_size, block_size) for _ in range(n_heads)])
        self.proj = nn.Linear(n_heads * head_size, n_embd)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)  # concat outputs along feature dim
        return self.proj(out)  # project back to n_embd dimensions

# Quick test
mha = MultiHeadAttention(n_embd=10, n_heads=2, head_size=8, block_size=3)
test_in = torch.randn(2, 3, 10)
test_out = mha(test_in)
print(f'Multi-head attention: input {test_in.shape} -> output {test_out.shape}')

## 7. Full Transformer Model

Now we assemble everything: token embeddings, positional embeddings,
transformer blocks (multi-head attention + feed-forward, each with
residual connections and layer norm), and a final linear head.

### Exercise 11: Build the Transformer

This is the most complex exercise. We need three classes:

**`FeedForward`:**
- A small MLP: `Linear(n_embd, 4*n_embd)` -> `ReLU` -> `Linear(4*n_embd, n_embd)`

**`TransformerBlock`:**
- Contains: `MultiHeadAttention`, `FeedForward`, two `LayerNorm` layers
- Forward: `x = x + sa(ln1(x))` then `x = x + ffwd(ln2(x))` (pre-norm residual connections)
- Note: `head_size = n_embd // n_heads`

**`CharTransformer`:**
- `token_emb`: `nn.Embedding(vocab_size, n_embd)` -- character embeddings
- `pos_emb`: `nn.Embedding(block_size, n_embd)` -- positional embeddings
- `blocks`: `nn.Sequential` of `n_layers` `TransformerBlock`s
- `ln_f`: Final `LayerNorm(n_embd)`
- `lm_head`: `nn.Linear(n_embd, vocab_size)` -- output projection
- Forward: `x = token_emb(idx) + pos_emb(arange(T))`, pass through blocks, layer norm, lm_head
- If targets provided, compute cross-entropy loss (flatten `(B,T,V)` to `(B*T, V)` and `(B*T,)`)
- `generate`: Autoregressively sample by cropping to `block_size`, forward, softmax last position, sample

**Tensor shapes to keep in mind:**
- Input `idx`: `(B, T)` -- batch of token index sequences
- After embeddings: `(B, T, n_embd)`
- After blocks: `(B, T, n_embd)`
- After lm_head: `(B, T, vocab_size)` -- logits

In [ ]:
# Exercise 11: Full Transformer

class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        # nn.Sequential chains layers -- like a pipeline: input -> Linear -> ReLU -> Linear -> output
        # YOUR CODE HERE: Sequential with Linear -> ReLU -> Linear
        # Expand to 4*n_embd in the hidden layer, then back to n_embd
        self.net = nn.Sequential(
            # YOUR CODE HERE
        )

    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_heads, block_size):
        super().__init__()
        head_size = n_embd // n_heads  # // is integer division (like Math.floor(a / b))
        # YOUR CODE HERE: create self-attention, feed-forward, and two layer norms
        self.sa = None    # MultiHeadAttention(...)
        self.ffwd = None  # FeedForward(...)
        self.ln1 = None   # nn.LayerNorm(n_embd)
        self.ln2 = None   # nn.LayerNorm(n_embd)

    def forward(self, x):
        # Pre-norm residual connections:
        # x = x + sublayer(norm(x))  -- the "+" is the residual/skip connection
        # YOUR CODE HERE: pre-norm residual connections
        # x = x + self_attention(layer_norm_1(x))
        # x = x + feed_forward(layer_norm_2(x))
        return x


class CharTransformer(nn.Module):
    def __init__(self, vocab_size, n_embd, n_heads, n_layers, block_size):
        super().__init__()
        self.block_size = block_size
        # YOUR CODE HERE: create embeddings, blocks, final layer norm, and output head
        self.token_emb = None  # nn.Embedding(vocab_size, n_embd) -- lookup table: index -> vector
        self.pos_emb = None    # nn.Embedding(block_size, n_embd) -- position -> vector
        self.blocks = None     # nn.Sequential of TransformerBlocks
        self.ln_f = None       # nn.LayerNorm(n_embd) -- final layer normalization
        self.lm_head = None    # nn.Linear(n_embd, vocab_size) -- projects to vocab logits

    def forward(self, idx, targets=None):  # targets=None: default parameter (like targets = null in JS)
        B, T = idx.shape
        # YOUR CODE HERE:
        # 1. Token embeddings: self.token_emb(idx) -> (B, T, n_embd)
        # 2. Position embeddings: self.pos_emb(torch.arange(T, device=idx.device)) -> (T, n_embd)
        #    device= ensures the tensor is on the same device (CPU/GPU) as idx
        # 3. Add them: x = tok_emb + pos_emb  (broadcasting adds pos_emb to each batch)
        # 4. Pass through self.blocks
        # 5. Final layer norm
        # 6. Output head -> logits of shape (B, T, vocab_size)
        tok_emb = None
        pos_emb = None
        x = None
        x = None  # blocks
        x = None  # layer norm
        logits = None  # lm_head

        if targets is None:  # None check: like if (targets === null) in JS
            loss = None
        else:
            # .view() reshapes tensor (like reinterpreting array dimensions)
            # Flatten logits from (B, T, V) to (B*T, V) and targets from (B, T) to (B*T)
            B, T, V = logits.shape
            logits_flat = logits.view(B * T, V)
            targets_flat = targets.view(B * T)
            loss = F.cross_entropy(logits_flat, targets_flat)
        return logits, loss  # return tuple (like returning [logits, loss] in JS)

    @torch.no_grad()  # decorator: wraps this method in torch.no_grad() context (disables gradients)
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # YOUR CODE HERE:
            # 1. Crop idx to last block_size tokens
            # idx[:, -self.block_size:] -- [:, ...] selects all rows, [-n:] selects last n columns
            # 2. Forward pass to get logits
            # 3. Take logits at the last time step: logits[:, -1, :] (all batches, last position, all vocab)
            # 4. Softmax to get probabilities
            # 5. Sample next token with torch.multinomial
            # 6. Append to idx with torch.cat along dim=1
            idx_cond = None  # crop to block_size
            logits, _ = self(idx_cond)  # _ discards the loss (throwaway variable)
            logits = None    # last time step only
            probs = None     # softmax
            idx_next = None  # sample
            idx = torch.cat([idx, idx_next], dim=1)  # like [...idx, idx_next] along time axis
        return idx


# Hyperparameters
tf_block_size = 8
tf_n_embd = 64
tf_n_heads = 4
tf_n_layers = 2

model = CharTransformer(vocab_size, tf_n_embd, tf_n_heads, tf_n_layers, tf_block_size)
# .numel() returns total number of elements; sum() with generator expr = total param count
print(f'Transformer parameters: {sum(p.numel() for p in model.parameters()):,}')  # :, adds thousand separators

In [ ]:
#@title Solution (click to expand)
class FeedForward(nn.Module):
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),  # expand
            nn.ReLU(),                        # activation
            nn.Linear(4 * n_embd, n_embd),   # project back
        )  # trailing comma is valid in Python (like JS)

    def forward(self, x):
        return self.net(x)  # nn.Sequential runs layers in order


class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_heads, block_size):
        super().__init__()
        head_size = n_embd // n_heads  # // = integer division (Math.floor(a/b) in JS)
        self.sa = MultiHeadAttention(n_embd, n_heads, head_size, block_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))   # residual connection: x + sublayer(norm(x))
        x = x + self.ffwd(self.ln2(x)) # second residual connection
        return x


class CharTransformer(nn.Module):
    def __init__(self, vocab_size, n_embd, n_heads, n_layers, block_size):
        super().__init__()
        self.block_size = block_size
        self.token_emb = nn.Embedding(vocab_size, n_embd)  # lookup table: token index -> embedding vector
        self.pos_emb = nn.Embedding(block_size, n_embd)    # position index -> embedding vector
        self.blocks = nn.Sequential(
            *[TransformerBlock(n_embd, n_heads, block_size) for _ in range(n_layers)]
        )  # *[...] unpacks list into Sequential args (like ...spread in JS)
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape  # unpack: like const [B, T] = idx.shape
        tok_emb = self.token_emb(idx)  # (B, T, n_embd)
        pos_emb = self.pos_emb(torch.arange(T, device=idx.device))  # (T, n_embd) -- broadcasts across batch
        x = tok_emb + pos_emb  # element-wise add (broadcasting handles the batch dimension)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)  # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, V = logits.shape
            logits_flat = logits.view(B * T, V)    # .view() reshapes (like reinterpreting dimensions)
            targets_flat = targets.view(B * T)      # flatten from (B, T) to (B*T,)
            loss = F.cross_entropy(logits_flat, targets_flat)
        return logits, loss

    @torch.no_grad()  # decorator: disables gradient tracking for this method (saves memory during inference)
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]  # [:, -n:] = all batches, last n tokens (like .slice(-n))
            logits, _ = self(idx_cond)  # self(...) calls forward(); _ discards the loss
            logits = logits[:, -1, :]  # take last time step only: (B, vocab_size)
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)  # sample one token per batch
            idx = torch.cat([idx, idx_next], dim=1)  # append along time dimension
        return idx


# Hyperparameters
tf_block_size = 8
tf_n_embd = 64
tf_n_heads = 4
tf_n_layers = 2

model = CharTransformer(vocab_size, tf_n_embd, tf_n_heads, tf_n_layers, tf_block_size)
print(f'Transformer parameters: {sum(p.numel() for p in model.parameters()):,}')

### Build the Transformer Dataset

Run this cell as-is. It creates input/target pairs for the transformer where each position predicts the next token.

> **Python for JS devs:** `seq[start:end]` is like `seq.slice(start, end)`. `[-1]` indexes from the end (last element). `[0] * n` creates an array of n zeros (like `new Array(n).fill(0)`). `max()` is `Math.max()`.

In [ ]:
# Build dataset for the transformer (input sequence -> next char at each position)
def build_transformer_dataset(name_list, block_size):
    X, Y = [], []
    for name in name_list:
        seq = [0] + [stoi[ch] for ch in name] + [0]  # list comprehension: like name.split('').map(ch => stoi[ch])
        for i in range(len(seq) - 1):
            # Grab up to block_size input chars and the corresponding targets
            start = max(0, i - block_size + 1)  # max() is like Math.max()
            inp = seq[start:i+1]     # slice: like seq.slice(start, i+1)
            tgt = seq[start+1:i+2]
            # Pad on the left if shorter than block_size
            pad_len = block_size - len(inp)
            inp = [0] * pad_len + inp  # [0]*n creates n zeros; + concatenates lists
            tgt = [0] * pad_len + tgt
            X.append(inp)
            Y.append(tgt)
    return torch.tensor(X), torch.tensor(Y)

Xtr_tf, Ytr_tf = build_transformer_dataset(train_names, tf_block_size)
Xval_tf, Yval_tf = build_transformer_dataset(val_names, tf_block_size)
print(f'Train: X={Xtr_tf.shape}, Y={Ytr_tf.shape}')
print(f'Val:   X={Xval_tf.shape}, Y={Yval_tf.shape}')

### Exercise 12: Train the Transformer

**Goal:** Train the transformer for 10000 steps total (5000 at lr=3e-3, then 5000 at lr=1e-3).

**Steps:**
1. Create an AdamW optimizer for `model.parameters()` with `lr=3e-3`
2. Training loop: sample mini-batch, forward pass (returns logits and loss), zero grad, backward, step
3. After 5000 steps, reduce learning rate to `1e-3` by setting `g_opt['lr']` for each param group
4. Train 5000 more steps
5. Evaluate on validation set

**Expected output:** Loss should decrease from ~3.3 to ~2.0

In [ ]:
# Exercise 12: Training loop for the transformer

# YOUR CODE HERE: Create optimizer
# AdamW is a popular optimizer (handles learning rate, momentum, weight decay automatically)
optimizer = None  # torch.optim.AdamW(model.parameters(), lr=3e-3)

tf_train_losses = []

# Phase 1: 5000 steps at lr=3e-3
for step in range(5000):
    # Mini-batch
    ix = torch.randint(0, Xtr_tf.shape[0], (64,))
    Xb, Yb = Xtr_tf[ix], Ytr_tf[ix]

    # YOUR CODE HERE: forward pass, zero grad, backward, optimizer step
    # model(Xb, Yb) returns (logits, loss) -- the model computes loss internally
    # optimizer.zero_grad(set_to_none=True) clears gradients (set_to_none is faster than zeroing)
    # loss.backward() computes gradients via autograd
    # optimizer.step() updates all parameters
    logits, loss = None, None

    if step % 1000 == 0:
        print(f'Step {step:5d} | loss: {loss.item():.4f}')
    tf_train_losses.append(loss.item())

# YOUR CODE HERE: Reduce learning rate to 1e-3
# optimizer.param_groups is a list of dicts; each has an 'lr' key you can mutate

# Phase 2: 5000 more steps at lr=1e-3
for step in range(5000):
    ix = torch.randint(0, Xtr_tf.shape[0], (64,))
    Xb, Yb = Xtr_tf[ix], Ytr_tf[ix]

    # YOUR CODE HERE: same as above
    logits, loss = None, None

    if step % 1000 == 0:
        print(f'Step {step+5000:5d} | loss: {loss.item():.4f}')
    tf_train_losses.append(loss.item())

# Validation loss
with torch.no_grad():
    _, tf_val_loss = model(Xval_tf, Yval_tf)  # _ discards logits, keep only loss
    print(f'\nTransformer val loss: {tf_val_loss.item():.4f}')

In [ ]:
#@title Solution (click to expand)
# Training loop for the transformer
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)  # 3e-3 = 0.003 (scientific notation, same as JS)
tf_train_losses = []

for step in range(5000):
    # mini-batch
    ix = torch.randint(0, Xtr_tf.shape[0], (64,))
    Xb, Yb = Xtr_tf[ix], Ytr_tf[ix]

    logits, loss = model(Xb, Yb)  # model(...) calls forward(), returns (logits, loss) tuple
    optimizer.zero_grad(set_to_none=True)  # clear old gradients
    loss.backward()   # autograd: compute d(loss)/d(param) for every parameter
    optimizer.step()  # update all params using Adam algorithm

    if step % 1000 == 0:
        print(f'Step {step:5d} | loss: {loss.item():.4f}')
    tf_train_losses.append(loss.item())

# Reduce learning rate and train more
for g_opt in optimizer.param_groups:  # param_groups is a list of config dicts
    g_opt['lr'] = 1e-3  # mutate the dict in place to change learning rate

for step in range(5000):
    ix = torch.randint(0, Xtr_tf.shape[0], (64,))
    Xb, Yb = Xtr_tf[ix], Ytr_tf[ix]
    logits, loss = model(Xb, Yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step % 1000 == 0:
        print(f'Step {step+5000:5d} | loss: {loss.item():.4f}')
    tf_train_losses.append(loss.item())

# Validation loss
with torch.no_grad():  # no gradients needed for evaluation
    _, tf_val_loss = model(Xval_tf, Yval_tf)
    print(f'\nTransformer val loss: {tf_val_loss.item():.4f}')

### Generate Names from the Transformer

Run this cell to see the transformer's generated names.

In [ ]:
# Generate names from the transformer
torch.manual_seed(42)
print('Transformer generated names:')
for _ in range(15):
    idx = torch.zeros((1, 1), dtype=torch.long)  # start with '.' token; dtype=torch.long = 64-bit integer
    generated = model.generate(idx, max_new_tokens=15)
    tokens = generated[0].tolist()  # .tolist() converts tensor to plain Python list (like Array.from())
    name = ''
    for t in tokens[1:]:  # [1:] = skip first element (like .slice(1))
        if t == 0:
            break
        name += itos[t]  # string concatenation (same as JS)
    print('  ' + name)

## 8. Comparison

Let us compare all models by their validation loss and look at sample outputs.

In [ ]:
# Summary of validation losses
print('=== Validation Losses (NLL) ===')
print(f'  Bigram (count):   {bigram_count_loss.item():.4f}')
print(f'  Neural Bigram:    {neural_bigram_loss.item():.4f}')
print(f'  MLP:              {mlp_val_loss.item():.4f}')
print(f'  Transformer:      {tf_val_loss.item():.4f}')

# Bar chart
model_names = ['Bigram\n(count)', 'Neural\nBigram', 'MLP', 'Transformer']
losses = [bigram_count_loss.item(), neural_bigram_loss.item(),
          mlp_val_loss.item(), tf_val_loss.item()]
colors = ['#4e79a7', '#f28e2b', '#e15759', '#76b7b2']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(model_names, losses, color=colors, edgecolor='black', linewidth=0.5)
ax.set_ylabel('Validation Loss (NLL)')
ax.set_title('Model Comparison: Validation Loss')
for bar, loss in zip(bars, losses):  # zip pairs corresponding elements from two lists
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f'{loss:.3f}', ha='center', va='bottom', fontweight='bold')
ax.set_ylim(0, max(losses) * 1.15)
plt.tight_layout()
plt.show()

In [ ]:
# Training loss curves (MLP and Transformer)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))  # tuple unpacking: const [ax1, ax2] = ...

# MLP loss curve (smoothed)
window = 100
# List comprehension with range step: [expr for i in range(start, stop, step)]
mlp_smooth = [sum(mlp_losses[i:i+window])/window
              for i in range(0, len(mlp_losses)-window, window)]
ax1.plot(range(0, len(mlp_losses)-window, window), mlp_smooth, color='#e15759')
ax1.set_title('MLP Training Loss')
ax1.set_xlabel('Step')
ax1.set_ylabel('Loss')

# Transformer loss curve (smoothed)
tf_smooth = [sum(tf_train_losses[i:i+window])/window
             for i in range(0, len(tf_train_losses)-window, window)]
ax2.plot(range(0, len(tf_train_losses)-window, window), tf_smooth, color='#76b7b2')
ax2.set_title('Transformer Training Loss')
ax2.set_xlabel('Step')
ax2.set_ylabel('Loss')

plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side sample names from each model
print('=== Sample Generated Names ===')
print()  # print() with no args = console.log('') -- prints blank line

# Bigram (count)
g = torch.Generator().manual_seed(123)
print('Bigram (count):')
for _ in range(5):
    out = []
    ix = 0
    while True:
        p = P[ix]
        ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
        if ix == 0:
            break
        out.append(itos[ix])
    print(f'  {"" .join([] )+ "".join(out)}')

# Neural Bigram
g = torch.Generator().manual_seed(123)
print('\nNeural Bigram:')  # \n in f-string = newline (same as JS)
for _ in range(5):
    out = []
    ix = 0
    while True:
        xenc = F.one_hot(torch.tensor([ix]), num_classes=vocab_size).float()
        logits = xenc @ W  # @ = matrix multiply
        p = F.softmax(logits, dim=1)
        ix = torch.multinomial(p, num_samples=1, generator=g).item()
        if ix == 0:
            break
        out.append(itos[ix])
    print(f'  {"".join(out)}')

# MLP
g = torch.Generator().manual_seed(123)
print('\nMLP:')
for _ in range(5):
    out = []
    context = [0] * block_size
    while True:
        emb = C[torch.tensor([context])]
        h = torch.tanh(emb.view(1, -1) @ W1 + b1)
        logits = h @ W2 + b2
        p = F.softmax(logits, dim=1)
        ix = torch.multinomial(p, num_samples=1, generator=g).item()
        if ix == 0:
            break
        out.append(itos[ix])
        context = context[1:] + [ix]
    print(f'  {"".join(out)}')

# Transformer
torch.manual_seed(123)
print('\nTransformer:')
for _ in range(5):
    idx = torch.zeros((1, 1), dtype=torch.long)
    generated = model.generate(idx, max_new_tokens=15)
    tokens = generated[0].tolist()
    name = ''
    for t in tokens[1:]:
        if t == 0:
            break
        name += itos[t]
    print(f'  {name}')

print('\nDone! Visit mlearn.dev for the full lesson.')